# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Note: you may need to restart the kernel to use updated packages.

You can now save your data files in: /Users/leonhsu/Projects/GH_AT3/36106/assignment/AT3/data


---
## Student Information

In [2]:
# <Student to fill this section>
group_name = "Group 29"
student_name = "Leon Hsu"
student_id = "26324145"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [6]:
# <Student to fill this section>

### 0.b Import Packages

In [7]:
# <Student to fill this section>
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [8]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "economics.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [9]:
# <Student to fill this section>

In [10]:
df.head()

,economic_date,inflation_rate,interest_rate,unemployment_rate,consumer_sentiment,economic_cycle,loan_stress_index,spending_factor,housing_price_index,equity_market_index,recession_flag
0,2020-01-01,2.27,2.49,5.06,97.95,Expansion,41.79,0.731,102.02,100.11,0
1,2020-02-01,2.24,2.64,5.14,97.43,Expansion,42.74,0.724,102.05,102.17,0
2,2020-03-01,2.17,2.60,5.17,97.63,Expansion,42.33,0.724,100.61,98.48,0
3,2020-04-01,2.09,2.50,5.20,97.96,Expansion,41.61,0.725,100.20,95.69,0
4,2020-05-01,2.31,2.47,5.20,97.32,Expansion,42.41,0.720,99.27,95.06,0


In [11]:
df.shape

(72, 11)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   economic_date        72 non-null     object 
 1   inflation_rate       72 non-null     float64
 2   interest_rate        72 non-null     float64
 3   unemployment_rate    72 non-null     float64
 4   consumer_sentiment   72 non-null     float64
 5   economic_cycle       72 non-null     object 
 6   loan_stress_index    72 non-null     float64
 7   spending_factor      72 non-null     float64
 8   housing_price_index  72 non-null     float64
 9   equity_market_index  72 non-null     float64
 10  recession_flag       72 non-null     int64  
dtypes: float64(8), int64(1), object(2)
memory usage: 6.3+ KB


In [13]:
df.describe()

,inflation_rate,interest_rate,unemployment_rate,consumer_sentiment,loan_stress_index,spending_factor,housing_price_index,equity_market_index,recession_flag
count,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.0
mean,2.301806,2.862361,5.083056,88.288472,43.850556,0.658736,116.139583,111.105556,0.0
std,0.315239,0.263270,0.152257,10.565051,1.748479,0.080597,13.131213,18.402310,0.0
min,1.670000,2.360000,4.590000,69.120000,40.360000,0.508000,97.800000,89.020000,0.0
25%,2.077500,2.677500,5.000000,86.192500,42.410000,0.644750,103.707500,95.405000,0.0
50%,2.290000,2.910000,5.110000,92.250000,43.890000,0.687500,114.825000,103.255000,0.0
75%,2.472500,3.012500,5.190000,94.475000,44.882500,0.718000,125.040000,124.830000,0.0
max,3.090000,3.440000,5.420000,98.990000,47.710000,0.746000,142.400000,146.210000,0.0


In [14]:
# <Student to fill this section>
dataset_insights = """
provide a detailed analysis on the dataset, its dimensions, information, issues, ...
"""

In [15]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### Helper Functions

In [16]:
# Chart helpers
# -------------
# Every chart below is built from three small, named pieces instead of one
# big function per chart type:
#   _finish      applies the title/size/axis styling shared by every chart
#   _bar_chart   draws one bar per category — used by every function below
#   _bin_counts  groups a numeric column into readable ranges (e.g. '108–434')
# Each public function (numeric_distribution, categorical_distribution, ...)
# only has to say *what* to plot; the pieces above handle *how* to draw it.

import numpy as np

UNIT = 'Months'          # what one row of `df` represents, used in axis labels
BAR_COLOR = '#2563eb'
AXIS_CFG = dict(domainColor='#9ca3af', tickColor='#9ca3af', labelColor='#000', titleColor='#000')


def _finish(chart, title, width, height):
    """Apply the title, size and axis styling shared by every chart in this notebook."""
    return (chart.properties(title=title, width=width, height=height)
                 .configure_view(strokeWidth=0)
                 .configure_axis(**AXIS_CFG))


def _bar_chart(data, category_field, value_field, *, category_title, value_title,
               sort_order, tooltip, color, horizontal, title, height=None):
    """One bar per category, laid out as columns, or as rows when horizontal=True."""
    if horizontal:
        encoding = dict(
            y=alt.Y(f'{category_field}:N', title=category_title, sort=sort_order, axis=alt.Axis(grid=False)),
            x=alt.X(f'{value_field}:Q', title=value_title, axis=alt.Axis(grid=False)),
        )
        width, height = 520, height or max(250, 18 * len(data))
    else:
        encoding = dict(
            x=alt.X(f'{category_field}:N', title=category_title, sort=sort_order, axis=alt.Axis(labelAngle=0, grid=False)),
            y=alt.Y(f'{value_field}:Q', title=value_title, axis=alt.Axis(grid=False)),
        )
        width, height = max(440, 70 * len(data)), height or 350

    chart = alt.Chart(data).mark_bar(color=color).encode(**encoding, tooltip=tooltip)
    return _finish(chart, title, width, height)


def _decimals_for_width(bin_width):
    """How many decimal places keep a bin label readable, given how wide the bin is."""
    if bin_width >= 5:
        return 0
    if bin_width >= 0.5:
        return 1
    return 2


def _bin_counts(series, bins, min_bins, max_bins):
    """Group a numeric series into readable bins (e.g. '108–434') and count rows per bin."""
    if bins == 'auto':
        # Freedman-Diaconis bin width, clipped to [min_bins, max_bins] so labels stay readable
        edges = np.histogram_bin_edges(series.dropna(), bins='fd')
        bins = int(np.clip(len(edges) - 1, min_bins, max_bins))

    intervals = pd.cut(series, bins=bins, include_lowest=True)
    counts = intervals.value_counts().sort_index().reset_index()
    counts.columns = ['interval', 'Count']

    decimals = _decimals_for_width(counts['interval'].iloc[0].length)
    label = lambda v: f'{v:,.{decimals}f}'
    counts['Bin'] = counts['interval'].apply(lambda iv: f'{label(iv.left)}–{label(iv.right)}')
    return counts[['Bin', 'Count']]


def numeric_distribution(col, bins='auto', discrete=False, min_bins=8, max_bins=20, data=None):
    """Bar chart of how many Months fall into each value (discrete=True) or range of `col`."""
    data = df if data is None else data
    if discrete:
        counts = data[col].value_counts().sort_index().reset_index()
        counts.columns = [col, 'Count']
        plot_data = pd.DataFrame({'Bin': counts[col].astype(str), 'Count': counts['Count']})
    else:
        plot_data = _bin_counts(data[col], bins, min_bins, max_bins)

    return _bar_chart(
        plot_data, 'Bin', 'Count', category_title=col, value_title=f'Number of {UNIT}',
        sort_order=list(plot_data['Bin']), horizontal=False, title=f'Distribution of {col}',
        color=BAR_COLOR, tooltip=[alt.Tooltip('Bin:N', title=col), alt.Tooltip('Count:Q', title=UNIT, format=',')],
    )


def categorical_distribution(col, order=None, top_n=None, horizontal=False, data=None):
    """Bar chart of how many Months fall into each category of `col` (optionally only the top_n)."""
    data = df if data is None else data
    counts = data[col].astype(str).value_counts().reset_index()
    counts.columns = [col, 'Count']
    if top_n is not None:
        counts = counts.head(top_n)
    counts['Share'] = counts['Count'] / len(data)

    sort_order = order if order is not None else ('-x' if horizontal else '-y')
    title = f'Distribution of {col}' if top_n is None else f'Top {top_n} {col} by {UNIT}'
    tooltip = [alt.Tooltip(f'{col}:N'),
               alt.Tooltip('Count:Q', title=UNIT, format=','),
               alt.Tooltip('Share:Q', format='.1%')]

    return _bar_chart(counts, col, 'Count', category_title=col, value_title=f'Number of {UNIT}',
                       sort_order=sort_order, horizontal=horizontal, title=title,
                       color=BAR_COLOR, tooltip=tooltip)


LINE_COLOR = '#9ca3af'
CYCLE_ORDER = ['Expansion', 'Peak', 'Slowdown', 'Recovery']


def time_series(col, data=None):
    """Monthly line of `col`, with points coloured by economic_cycle to show regime shifts."""
    data = df if data is None else data
    plot_data = data[['economic_date', 'economic_cycle', col]].copy()
    plot_data['economic_date'] = pd.to_datetime(plot_data['economic_date'])

    base = alt.Chart(plot_data).encode(
        x=alt.X('economic_date:T', title='Month'),
        y=alt.Y(f'{col}:Q', title=col, scale=alt.Scale(zero=False)),
    )
    line = base.mark_line(color=LINE_COLOR)
    points = base.mark_circle(size=45, opacity=1).encode(
        color=alt.Color('economic_cycle:N', title='Economic cycle', sort=CYCLE_ORDER),
        tooltip=[alt.Tooltip('economic_date:T', format='%b %Y'), 'economic_cycle:N', alt.Tooltip(f'{col}:Q')],
    )
    return _finish(line + points, f'{col} over Time', 620, 300)


def cycle_summary(col, data=None):
    """Mean, min and max of `col` within each economic cycle, in chronological order."""
    data = df if data is None else data
    return data.groupby('economic_cycle')[col].agg(['mean', 'min', 'max']).round(3).reindex(CYCLE_ORDER)

> All 11 columns are explored below. The table has no identifier, personal or location fields, as each row describes one month of the economy rather than a customer.

In [17]:
df_cleaned = df.copy()

### B.2 Explore Feature of Interest `economic_date`

> You can add more cells in this section

In [18]:
economic_date_df = df['economic_date']

In [19]:
economic_date_df.describe()

count             72
unique            72
top       2020-01-01
freq               1
Name: economic_date, dtype: object

In [20]:
economic_date_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: economic_date
Non-Null Count  Dtype 
--------------  ----- 
72 non-null     object
dtypes: object(1)
memory usage: 708.0+ bytes


In [21]:
economic_date_df.unique()

array(['2020-01-01', '2020-02-01', '2020-03-01', '2020-04-01',
       '2020-05-01', '2020-06-01', '2020-07-01', '2020-08-01',
       '2020-09-01', '2020-10-01', '2020-11-01', '2020-12-01',
       '2021-01-01', '2021-02-01', '2021-03-01', '2021-04-01',
       '2021-05-01', '2021-06-01', '2021-07-01', '2021-08-01',
       '2021-09-01', '2021-10-01', '2021-11-01', '2021-12-01',
       '2022-01-01', '2022-02-01', '2022-03-01', '2022-04-01',
       '2022-05-01', '2022-06-01', '2022-07-01', '2022-08-01',
       '2022-09-01', '2022-10-01', '2022-11-01', '2022-12-01',
       '2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01',
       '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01',
       '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01',
       '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01',
       '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01',
       '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01',
       '2025-01-01', '2025-02-01', '2025-03-01', '2025-

In [22]:
economic_date_df.isna().sum()

np.int64(0)

In [23]:
# Stored as text (object), so convert to datetime for analysis
economic_date_dt = pd.to_datetime(df['economic_date'], errors='coerce')
print('Unparseable dates:', economic_date_dt.isna().sum())
print('Earliest:', economic_date_dt.min().date(), '| Latest:', economic_date_dt.max().date())
print('Days of month used:', economic_date_dt.dt.day.unique())
print('Duplicate months:', economic_date_dt.duplicated().sum())
print('Months per year:', economic_date_dt.dt.year.value_counts().sort_index().to_dict())

Unparseable dates: 0
Earliest: 2020-01-01 | Latest: 2025-12-01
Days of month used: [1]
Duplicate months: 0
Months per year: {2020: 12, 2021: 12, 2022: 12, 2023: 12, 2024: 12, 2025: 12}


In [24]:
# Gap between consecutive records (days), to confirm there are no missing months
economic_date_dt.diff().dt.days.describe()

count    71.000000
mean     30.436620
std       0.805914
min      28.000000
25%      30.000000
50%      31.000000
75%      31.000000
max      31.000000
Name: economic_date, dtype: float64

In [25]:
# <Student to fill this section>
feature_1_insights = """
`economic_date` is the reference month for each set of economic indicators. It is stored as text (object) and
needs converting to datetime, and all 72 values parse cleanly with no duplicates or missing values. The data
covers January 2020 to December 2025, with 12 months each year and no gaps between consecutive records.

Every date is the first day of the month, whereas the balance history uses month-end dates (e.g. 2022-01-31) and
transactions use full timestamps. Joining on the raw date would therefore fail; all three should be converted
to a common monthly period (e.g. `to_period('M')`) before merging. The first 24 months (2020–2021) fall before the
balance history (from 2022) and transactions (from 2023) begin, so they are only useful as lagged context.

With one row per month, every customer shares the same economic values in a given month. These features
describe time rather than customers, so they cannot separate one customer's risk from another's within the
same month, and there are only 72 observations from which to learn any economic effect.
"""

In [26]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `inflation_rate`

> You can add more cells in this section

In [27]:
inflation_rate_df = df['inflation_rate']

In [28]:
inflation_rate_df.describe()

count    72.000000
mean      2.301806
std       0.315239
min       1.670000
25%       2.077500
50%       2.290000
75%       2.472500
max       3.090000
Name: inflation_rate, dtype: float64

In [29]:
inflation_rate_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: inflation_rate
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [30]:
inflation_rate_df.unique()

array([2.27, 2.24, 2.17, 2.09, 2.31, 2.32, 2.23, 2.05, 2.16, 2.1 , 2.37,
       2.43, 2.55, 2.52, 2.44, 2.46, 2.02, 2.08, 1.8 , 1.84, 1.96, 2.29,
       2.06, 2.48, 2.22, 2.15, 2.28, 2.41, 2.47, 2.38, 2.58, 2.93, 3.09,
       2.98, 3.08, 2.96, 2.91, 2.72, 2.65, 2.51, 2.54, 2.61, 2.62, 2.33,
       2.3 , 2.19, 2.07, 1.89, 1.87, 1.74, 1.78, 1.82, 1.67])

In [31]:
inflation_rate_df.isna().sum()

np.int64(0)

In [32]:
print('Skewness:', round(df['inflation_rate'].skew(), 2))
print('Highest:', df.loc[df['inflation_rate'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['inflation_rate'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['inflation_rate'].diff().abs().max(), 3))

Skewness: 0.45
Highest: 2023-10-01 | Lowest: 2025-12-01
Largest month-on-month move: 0.35


In [33]:
numeric_distribution('inflation_rate', min_bins=6, max_bins=12)

alt.Chart(...)

In [34]:
time_series('inflation_rate')

alt.LayerChart(...)

In [35]:
cycle_summary('inflation_rate')

,mean,min,max
economic_cycle,,,
Expansion,2.262,2.05,2.55
Peak,2.199,1.80,2.48
Slowdown,2.657,2.32,3.09
Recovery,2.089,1.67,2.62


In [36]:
# <Student to fill this section>
feature_2_insights = """
`inflation_rate` ranges from 1.67% to 3.09% with a mean of 2.30%, a median of 2.29%, a standard deviation of 0.32
and no missing values. Its distribution is fairly symmetric (skewness 0.45), and it peaks at 3.09% in October
2023 during the Slowdown phase before easing to its lowest value of 1.67% in December 2025.

The series stays inside a narrow band close to a typical 2–3% inflation target, and it does not reflect actual
Australian conditions over 2020–2025, when CPI inflation rose to around 7.8% in late 2022. The data should
therefore be treated as a synthetic scenario rather than real economic history.

Inflation is highest on average in the Slowdown phase (2.66%) and moves against `consumer_sentiment`
(r = -0.64) and `spending_factor` (r = -0.63), while rising with `loan_stress_index` (r = 0.54), which is a
plausible link between cost-of-living pressure and borrower stress. Because it barely varies and is shared by
every customer in a month, it is best used as background context or as a lagged feature for loan stress, rather
than as a direct predictor of individual behaviour.
"""

In [37]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `interest_rate`

> You can add more cells in this section

In [38]:
interest_rate_df = df['interest_rate']

In [39]:
interest_rate_df.describe()

count    72.000000
mean      2.862361
std       0.263270
min       2.360000
25%       2.677500
50%       2.910000
75%       3.012500
max       3.440000
Name: interest_rate, dtype: float64

In [40]:
interest_rate_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: interest_rate
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [41]:
interest_rate_df.unique()

array([2.49, 2.64, 2.6 , 2.5 , 2.47, 2.36, 2.54, 2.56, 2.58, 2.41, 2.39,
       2.55, 2.59, 2.68, 2.77, 2.8 , 2.76, 2.95, 2.94, 2.85, 2.75, 2.81,
       2.69, 2.72, 2.91, 2.99, 3.07, 3.04, 3.01, 3.06, 3.17, 3.09, 2.73,
       2.67, 2.89, 3.05, 2.93, 2.98, 3.02, 2.96, 3.2 , 3.27, 3.33, 3.44,
       3.35, 3.3 , 3.15])

In [42]:
interest_rate_df.isna().sum()

np.int64(0)

In [43]:
print('Skewness:', round(df['interest_rate'].skew(), 2))
print('Highest:', df.loc[df['interest_rate'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['interest_rate'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['interest_rate'].diff().abs().max(), 3))

Skewness: 0.05
Highest: 2025-06-01 | Lowest: 2020-06-01
Largest month-on-month move: 0.21


In [44]:
numeric_distribution('interest_rate', min_bins=6, max_bins=12)

alt.Chart(...)

In [45]:
time_series('interest_rate')

alt.LayerChart(...)

In [46]:
cycle_summary('interest_rate')

,mean,min,max
economic_cycle,,,
Expansion,2.518,2.36,2.68
Peak,2.854,2.68,3.07
Slowdown,2.926,2.67,3.17
Recovery,3.151,2.91,3.44


In [47]:
# <Student to fill this section>
feature_3_insights = """
`interest_rate` ranges from 2.36% to 3.44% with a mean of 2.86%, a median of 2.91%, a standard deviation of 0.26
and no missing values. It is almost perfectly symmetric (skewness 0.05), with no month moving by more than 0.21
percentage points.

Rather than following the economic cycle, the rate drifts upward over the whole period, from its low of 2.36% in
June 2020 to a high of 3.44% in June 2025, with mean rates rising from 2.52% (Expansion) to 3.15% (Recovery).
This does not match Australia's cash rate history, which fell to 0.10% in 2020 and then rose sharply to 4.35%
by late 2023, so the steep 2022–2023 tightening cycle, a major driver of real mortgage stress, is absent.

Because it trends steadily with time, it is strongly correlated with other trending series such as
`housing_price_index` (r = 0.87) and `equity_market_index` (r = 0.73). These relationships largely reflect the
shared trend rather than an economic link, so it is a risk of spurious correlation if several of these
indicators are used together, and differencing (month-on-month change) would be more meaningful than the level.
"""

In [48]:
# Do not modify this code
print_tile(size="h3", key='feature_3_insights', value=feature_3_insights)

### B.5 Explore Feature of Interest `unemployment_rate`

> You can add more cells in this section

In [49]:
unemployment_rate_df = df['unemployment_rate']

In [50]:
unemployment_rate_df.describe()

count    72.000000
mean      5.083056
std       0.152257
min       4.590000
25%       5.000000
50%       5.110000
75%       5.190000
max       5.420000
Name: unemployment_rate, dtype: float64

In [51]:
unemployment_rate_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: unemployment_rate
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [52]:
unemployment_rate_df.unique()

array([5.06, 5.14, 5.17, 5.2 , 5.24, 5.04, 5.03, 5.07, 4.93, 5.  , 5.18,
       5.11, 5.13, 5.1 , 5.12, 5.15, 5.09, 4.98, 4.95, 4.79, 4.66, 4.59,
       4.64, 4.75, 4.94, 5.08, 4.97, 4.99, 5.16, 5.26, 5.23, 5.3 , 5.19,
       5.22, 5.27, 5.42, 5.34, 5.02, 4.88, 5.05])

In [53]:
unemployment_rate_df.isna().sum()

np.int64(0)

In [54]:
print('Skewness:', round(df['unemployment_rate'].skew(), 2))
print('Highest:', df.loc[df['unemployment_rate'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['unemployment_rate'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['unemployment_rate'].diff().abs().max(), 3))

Skewness: -1.09
Highest: 2024-03-01 | Lowest: 2022-07-01
Largest month-on-month move: 0.32


In [55]:
numeric_distribution('unemployment_rate', min_bins=6, max_bins=12)

alt.Chart(...)

In [56]:
time_series('unemployment_rate')

alt.LayerChart(...)

In [57]:
cycle_summary('unemployment_rate')

,mean,min,max
economic_cycle,,,
Expansion,5.104,4.93,5.24
Peak,4.952,4.59,5.15
Slowdown,5.152,4.88,5.42
Recovery,5.124,4.93,5.22


In [58]:
# <Student to fill this section>
feature_4_insights = """
`unemployment_rate` ranges from 4.59% to 5.42% with a mean of 5.08%, a median of 5.11% and no missing values.
It has the smallest relative spread of any indicator (standard deviation 0.15) and is left-skewed (skewness
-1.09) because of a short dip to 4.59% in July 2022, with a high of 5.42% in March 2024.

The series is almost flat around 5%, with little difference across the economic cycle (mean 4.95% at Peak to
5.15% in Slowdown). It does not reflect the 2020 COVID shock, when Australian unemployment rose to about 7.5%,
or the tight labour market that followed, when it fell below 4%; 2020 is instead labelled an Expansion with
stable unemployment.

Unemployment is a key driver of credit stress in practice, and here it has a moderate positive correlation with
`loan_stress_index` (r = 0.49). However, with such a small range and only 72 values, it offers very
limited variation for a model to learn from, so its usefulness as a predictor is low.
"""

In [59]:
# Do not modify this code
print_tile(size="h3", key='feature_4_insights', value=feature_4_insights)

### B.6 Explore Feature of Interest `consumer_sentiment`

> You can add more cells in this section

In [60]:
consumer_sentiment_df = df['consumer_sentiment']

In [61]:
consumer_sentiment_df.describe()

count    72.000000
mean     88.288472
std      10.565051
min      69.120000
25%      86.192500
50%      92.250000
75%      94.475000
max      98.990000
Name: consumer_sentiment, dtype: float64

In [62]:
consumer_sentiment_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: consumer_sentiment
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [63]:
consumer_sentiment_df.unique()

array([97.95, 97.43, 97.63, 97.96, 97.32, 97.35, 97.26, 98.55, 98.23,
       98.22, 98.42, 98.3 , 98.99, 97.99, 97.54, 97.27, 96.71, 96.62,
       91.66, 92.16, 93.01, 92.38, 93.23, 93.57, 92.83, 92.27, 93.38,
       92.97, 93.25, 93.67, 93.76, 93.52, 93.34, 92.23, 92.82, 91.87,
       72.19, 71.8 , 71.74, 71.69, 70.98, 71.09, 70.23, 70.13, 69.19,
       69.12, 69.39, 69.43, 69.61, 70.12, 69.65, 69.9 , 71.62, 72.43,
       91.61, 91.6 , 90.78, 91.45, 91.88, 91.89, 92.08, 92.  , 91.55,
       91.53, 91.7 , 91.69, 91.85, 92.21, 92.43, 93.18, 92.4 ])

In [64]:
consumer_sentiment_df.isna().sum()

np.int64(0)

In [65]:
print('Skewness:', round(df['consumer_sentiment'].skew(), 2))
print('Highest:', df.loc[df['consumer_sentiment'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['consumer_sentiment'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['consumer_sentiment'].diff().abs().max(), 3))

Skewness: -1.02
Highest: 2021-01-01 | Lowest: 2023-10-01
Largest month-on-month move: 19.68


In [66]:
numeric_distribution('consumer_sentiment', min_bins=6, max_bins=12)

alt.Chart(...)

In [67]:
time_series('consumer_sentiment')

alt.LayerChart(...)

In [68]:
cycle_summary('consumer_sentiment')

,mean,min,max
economic_cycle,,,
Expansion,97.763,96.62,98.99
Peak,92.884,91.66,93.76
Slowdown,70.573,69.12,72.43
Recovery,91.933,90.78,93.18


In [69]:
# <Student to fill this section>
feature_5_insights = """
`consumer_sentiment` ranges from 69.12 to 98.99 with a mean of 88.29, a median of 92.25, a standard deviation of
10.57 and no missing values. It is the indicator most clearly tied to the economic cycle.

Instead of moving gradually, sentiment shifts in steps when the cycle label changes: it averages 97.76 in
Expansion, 92.88 at Peak, 70.57 in Slowdown and 91.93 in Recovery. It falls by 19.7 points in a single month
(December 2022 to January 2023) as the Slowdown begins, and jumps back by a similar amount in July 2024. This
produces a bimodal, left-skewed distribution (skewness -1.02), with one cluster near 70 and another above 90,
and shows that sentiment was generated from the cycle label rather than measured.

It is almost identical to `spending_factor` (r = 0.997) and strongly opposed to `loan_stress_index` (r = -0.78),
so these features carry largely the same information. Only one of `consumer_sentiment`, `spending_factor` and
`economic_cycle` should be kept to avoid multicollinearity.
"""

In [70]:
# Do not modify this code
print_tile(size="h3", key='feature_5_insights', value=feature_5_insights)

### B.7 Explore Feature of Interest `economic_cycle`

> You can add more cells in this section

In [71]:
economic_cycle_df = df['economic_cycle']

In [72]:
economic_cycle_df.describe()

count            72
unique            4
top       Expansion
freq             18
Name: economic_cycle, dtype: object

In [73]:
economic_cycle_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: economic_cycle
Non-Null Count  Dtype 
--------------  ----- 
72 non-null     object
dtypes: object(1)
memory usage: 708.0+ bytes


In [74]:
economic_cycle_df.unique()

array(['Expansion', 'Peak', 'Slowdown', 'Recovery'], dtype=object)

In [75]:
economic_cycle_df.isna().sum()

np.int64(0)

In [76]:
categorical_distribution('economic_cycle', order=CYCLE_ORDER)

alt.Chart(...)

In [77]:
# First and last month of each phase
(df.assign(economic_date=pd.to_datetime(df['economic_date']))
   .groupby('economic_cycle')['economic_date'].agg(['min', 'max', 'count']).reindex(CYCLE_ORDER))

,min,max,count
economic_cycle,,,
Expansion,2020-01-01,2021-06-01,18
Peak,2021-07-01,2022-12-01,18
Slowdown,2023-01-01,2024-06-01,18
Recovery,2024-07-01,2025-12-01,18


In [78]:
# Mean of every indicator within each phase
df.groupby('economic_cycle').mean(numeric_only=True).round(2).reindex(CYCLE_ORDER).T

economic_cycle,Expansion,Peak,Slowdown,Recovery
inflation_rate,2.26,2.20,2.66,2.09
interest_rate,2.52,2.85,2.93,3.15
unemployment_rate,5.10,4.95,5.15,5.12
consumer_sentiment,97.76,92.88,70.57,91.93
loan_stress_index,42.05,42.88,45.86,44.61
spending_factor,0.73,0.70,0.52,0.68
housing_price_index,100.67,110.29,119.39,134.21
equity_market_index,96.12,95.38,115.53,137.39
recession_flag,0.00,0.00,0.00,0.00


In [79]:
# <Student to fill this section>
feature_6_insights = """
`economic_cycle` labels each month with one of four phases, and each phase covers exactly 18 consecutive months
with no missing values: Expansion (January 2020 – June 2021), Peak (July 2021 – December 2022), Slowdown
(January 2023 – June 2024) and Recovery (July 2024 – December 2025).

This is a single, evenly split, scripted cycle rather than an observed one. There is no Contraction or Recession
phase, which is consistent with `recession_flag` being 0 throughout, and 2020 is labelled an Expansion even
though it was a recession year for Australia.

The label drives several other indicators: the Slowdown phase has by far the lowest `consumer_sentiment` (70.57)
and `spending_factor` (0.52) and the highest `loan_stress_index` (45.86) and inflation (2.66%). Because each phase
is one continuous block of time, the cycle is also a proxy for the calendar: transactions (2023–2025) only
cover Slowdown and Recovery, and the balance history adds Peak, so any cycle effect in a model cannot be
separated from a simple time trend. It is useful as an interpretable, one-hot encoded context feature, but its
effect should not be over-interpreted.
"""

In [80]:
# Do not modify this code
print_tile(size="h3", key='feature_6_insights', value=feature_6_insights)

### B.8 Explore Feature of Interest `loan_stress_index`

> You can add more cells in this section

In [81]:
loan_stress_index_df = df['loan_stress_index']

In [82]:
loan_stress_index_df.describe()

count    72.000000
mean     43.850556
std       1.748479
min      40.360000
25%      42.410000
50%      43.890000
75%      44.882500
max      47.710000
Name: loan_stress_index, dtype: float64

In [83]:
loan_stress_index_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: loan_stress_index
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [84]:
loan_stress_index_df.unique()

array([41.79, 42.74, 42.33, 41.61, 42.41, 42.05, 42.61, 41.19, 41.68,
       41.44, 41.17, 41.18, 40.36, 41.54, 42.51, 42.93, 43.57, 43.86,
       44.12, 43.49, 42.29, 43.55, 42.11, 42.69, 43.28, 41.96, 42.37,
       41.84, 41.89, 42.58, 42.85, 44.27, 43.3 , 44.38, 43.9 , 44.54,
       44.78, 44.63, 45.56, 45.46, 46.76, 46.73, 47.46, 47.71, 47.4 ,
       46.84, 46.51, 45.81, 46.37, 46.43, 44.04, 44.65, 44.79, 45.82,
       44.81, 44.2 , 44.36, 43.88, 44.35, 45.1 , 45.33, 45.36, 45.53,
       44.47, 43.05, 44.32, 43.59])

In [85]:
loan_stress_index_df.isna().sum()

np.int64(0)

In [86]:
print('Skewness:', round(df['loan_stress_index'].skew(), 2))
print('Highest:', df.loc[df['loan_stress_index'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['loan_stress_index'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['loan_stress_index'].diff().abs().max(), 3))

Skewness: 0.27
Highest: 2023-10-01 | Lowest: 2021-01-01
Largest month-on-month move: 1.89


In [87]:
numeric_distribution('loan_stress_index', min_bins=6, max_bins=12)

alt.Chart(...)

In [88]:
time_series('loan_stress_index')

alt.LayerChart(...)

In [89]:
cycle_summary('loan_stress_index')

,mean,min,max
economic_cycle,,,
Expansion,42.054,40.36,43.86
Peak,42.883,41.84,44.38
Slowdown,45.859,43.90,47.71
Recovery,44.606,43.05,45.82


In [90]:
# <Student to fill this section>
feature_7_insights = """
`loan_stress_index` ranges from 40.36 to 47.71 with a mean of 43.85, a median of 43.89, a standard deviation of
1.75 and no missing values. It is roughly symmetric (skewness 0.27), and no month moves by more than 1.89 points.

The index is lowest in January 2021 and highest in October 2023, and its phase averages rise from 42.05 in
Expansion and 42.88 at Peak to 45.86 in Slowdown, staying elevated at 44.61 in Recovery. It is the indicator
most connected to the others: it moves against `spending_factor` (r = -0.80) and `consumer_sentiment`
(r = -0.78), and with `interest_rate` (r = 0.58), `inflation_rate` (r = 0.54) and `unemployment_rate`
(r = 0.49), which is the pattern expected of a composite household-stress measure.

The scale is not documented, so a value of 45 has no absolute meaning, only a relative one. Because it summarises
several other indicators, it is a good single macro feature for loan default or financial stress models, and
using it instead of its components would limit multicollinearity. It is likely to be less relevant to AML
risk, which depends on customer and transaction behaviour rather than the economy.
"""

In [91]:
# Do not modify this code
print_tile(size="h3", key='feature_7_insights', value=feature_7_insights)

### B.9 Explore Feature of Interest `spending_factor`

> You can add more cells in this section

In [92]:
spending_factor_df = df['spending_factor']

In [93]:
spending_factor_df.describe()

count    72.000000
mean      0.658736
std       0.080597
min       0.508000
25%       0.644750
50%       0.687500
75%       0.718000
max       0.746000
Name: spending_factor, dtype: float64

In [94]:
spending_factor_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: spending_factor
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [95]:
spending_factor_df.unique()

array([0.731, 0.724, 0.725, 0.72 , 0.718, 0.737, 0.735, 0.73 , 0.746,
       0.734, 0.716, 0.683, 0.685, 0.693, 0.688, 0.694, 0.702, 0.69 ,
       0.701, 0.7  , 0.709, 0.719, 0.722, 0.712, 0.699, 0.542, 0.54 ,
       0.539, 0.538, 0.528, 0.527, 0.518, 0.508, 0.512, 0.513, 0.519,
       0.536, 0.548, 0.687, 0.677, 0.679, 0.684, 0.681, 0.678, 0.68 ,
       0.697, 0.692])

In [96]:
spending_factor_df.isna().sum()

np.int64(0)

In [97]:
print('Skewness:', round(df['spending_factor'].skew(), 2))
print('Highest:', df.loc[df['spending_factor'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['spending_factor'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['spending_factor'].diff().abs().max(), 3))

Skewness: -1.03
Highest: 2021-01-01 | Lowest: 2023-09-01
Largest month-on-month move: 0.143


In [98]:
numeric_distribution('spending_factor', min_bins=6, max_bins=12)

alt.Chart(...)

In [99]:
time_series('spending_factor')

alt.LayerChart(...)

In [100]:
cycle_summary('spending_factor')

,mean,min,max
economic_cycle,,,
Expansion,0.728,0.716,0.746
Peak,0.699,0.683,0.722
Slowdown,0.524,0.508,0.548
Recovery,0.684,0.677,0.697


In [101]:
# <Student to fill this section>
feature_8_insights = """
`spending_factor` ranges from 0.508 to 0.746 with a mean of 0.659, a median of 0.688, a standard deviation of
0.081 and no missing values. Its definition and unit are not documented, but it behaves like a 0–1 multiplier
on consumer spending.

Its shape is almost an exact copy of `consumer_sentiment`: it is left-skewed (skewness -1.03), averages 0.73 in
Expansion, 0.70 at Peak, 0.52 in Slowdown and 0.68 in Recovery, and drops by 0.143 in the same month that
sentiment collapses (January 2023). The correlation between the two is r = 0.997, so `spending_factor` is
effectively a rescaled version of sentiment.

In principle, a spending factor could explain changes in transaction volume and amounts, but the transactions
table shows almost no change in volume over time, so this link does not appear to have been built into the data.
Given that, `spending_factor` adds no information beyond `consumer_sentiment` and one of the two should be
dropped.
"""

In [102]:
# Do not modify this code
print_tile(size="h3", key='feature_8_insights', value=feature_8_insights)

### B.10 Explore Feature of Interest `housing_price_index`

> You can add more cells in this section

In [103]:
housing_price_index_df = df['housing_price_index']

In [104]:
housing_price_index_df.describe()

count     72.000000
mean     116.139583
std       13.131213
min       97.800000
25%      103.707500
50%      114.825000
75%      125.040000
max      142.400000
Name: housing_price_index, dtype: float64

In [105]:
housing_price_index_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: housing_price_index
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [106]:
housing_price_index_df.unique()

array([102.02, 102.05, 100.61, 100.2 ,  99.27,  99.17,  98.62,  97.8 ,
        98.  ,  98.83,  99.92, 100.76, 100.05, 100.92, 103.  , 103.61,
       103.59, 103.56, 103.74, 104.26, 103.95, 104.73, 107.84, 109.61,
       108.62, 109.27, 110.34, 111.79, 112.55, 113.7 , 114.53, 113.74,
       113.38, 113.67, 114.62, 114.89, 116.02, 115.62, 114.76, 115.4 ,
       117.19, 118.38, 119.74, 119.76, 118.43, 118.17, 117.99, 117.8 ,
       119.41, 122.12, 123.99, 125.16, 124.5 , 124.58, 125.21, 125.92,
       125.  , 125.97, 127.67, 129.35, 130.78, 132.15, 133.89, 134.99,
       138.49, 140.05, 139.8 , 141.08, 141.07, 140.76, 141.24, 142.4 ])

In [107]:
housing_price_index_df.isna().sum()

np.int64(0)

In [108]:
print('Skewness:', round(df['housing_price_index'].skew(), 2))
print('Highest:', df.loc[df['housing_price_index'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['housing_price_index'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['housing_price_index'].diff().abs().max(), 3))

Skewness: 0.44
Highest: 2025-12-01 | Lowest: 2020-08-01
Largest month-on-month move: 3.5


In [109]:
numeric_distribution('housing_price_index', min_bins=6, max_bins=12)

alt.Chart(...)

In [110]:
time_series('housing_price_index')

alt.LayerChart(...)

In [111]:
cycle_summary('housing_price_index')

,mean,min,max
economic_cycle,,,
Expansion,100.666,97.80,103.61
Peak,110.291,103.74,114.89
Slowdown,119.390,114.76,125.16
Recovery,134.212,125.00,142.40


In [112]:
# <Student to fill this section>
feature_9_insights = """
`housing_price_index` ranges from 97.80 to 142.40 with a mean of 116.14, a median of 114.83, a standard deviation
of 13.13 and no missing values. Starting near 100 in January 2020 (consistent with an index base of 100), it dips
to its low in August 2020 and then rises almost continuously to its high of 142.40 in December 2025, an
increase of about 40%.

The series is dominated by its upward trend: its correlation with time is r = 0.98, and every value is unique.
It keeps rising through the Slowdown phase (from 116 to 125), so it does not respond to the economic cycle, and
it is strongly correlated with the other trending series, `equity_market_index` (r = 0.93) and `interest_rate`
(r = 0.87). These correlations reflect the shared trend rather than a meaningful relationship.

In a model, the raw level acts as a proxy for the calendar date, which risks the model learning time rather than
economics, and new months would always fall outside the training range. Year-on-year or monthly percentage
change would be a more useful form, for example as context for mortgage or property-related lending risk.
"""

In [113]:
# Do not modify this code
print_tile(size="h3", key='feature_9_insights', value=feature_9_insights)

### B.11 Explore Feature of Interest `equity_market_index`

> You can add more cells in this section

In [114]:
equity_market_index_df = df['equity_market_index']

In [115]:
equity_market_index_df.describe()

count     72.000000
mean     111.105556
std       18.402310
min       89.020000
25%       95.405000
50%      103.255000
75%      124.830000
max      146.210000
Name: equity_market_index, dtype: float64

In [116]:
equity_market_index_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: equity_market_index
Non-Null Count  Dtype  
--------------  -----  
72 non-null     float64
dtypes: float64(1)
memory usage: 708.0 bytes


In [117]:
equity_market_index_df.unique()

array([100.11, 102.17,  98.48,  95.69,  95.06,  95.03,  97.65,  98.82,
        95.86,  92.3 ,  95.33,  98.32, 101.01, 100.08,  94.23,  90.21,
        89.02,  90.78,  90.53,  90.63,  90.9 ,  91.37,  91.57,  93.93,
        95.96,  95.43,  93.9 ,  91.67,  92.91,  94.25,  95.6 ,  97.84,
       100.88,  99.68, 101.07, 108.65, 108.8 , 105.44, 106.64, 104.34,
       101.48, 111.96, 111.86, 112.87, 112.33, 115.03, 122.  , 125.45,
       122.56, 124.66, 120.8 , 123.99, 124.07, 125.34, 123.15, 122.82,
       130.23, 129.12, 131.17, 134.15, 135.02, 135.29, 140.65, 141.18,
       138.61, 141.76, 146.21, 145.17, 144.48, 144.72, 145.66, 143.67])

In [118]:
equity_market_index_df.isna().sum()

np.int64(0)

In [119]:
print('Skewness:', round(df['equity_market_index'].skew(), 2))
print('Highest:', df.loc[df['equity_market_index'].idxmax(), 'economic_date'], '| Lowest:', df.loc[df['equity_market_index'].idxmin(), 'economic_date'])
print('Largest month-on-month move:', round(df['equity_market_index'].diff().abs().max(), 3))

Skewness: 0.6
Highest: 2025-07-01 | Lowest: 2021-05-01
Largest month-on-month move: 10.48


In [120]:
numeric_distribution('equity_market_index', min_bins=6, max_bins=12)

alt.Chart(...)

In [121]:
time_series('equity_market_index')

alt.LayerChart(...)

In [122]:
cycle_summary('equity_market_index')

,mean,min,max
economic_cycle,,,
Expansion,96.119,89.02,102.17
Peak,95.376,90.53,108.65
Slowdown,115.534,101.48,125.45
Recovery,137.392,122.82,146.21


In [123]:
# <Student to fill this section>
feature_10_insights = """
`equity_market_index` ranges from 89.02 to 146.21 with a mean of 111.11, a median of 103.26, a standard deviation of
18.40 and no missing values. It is the most volatile of the two market indices, with a largest single monthly move of 10.48
points (June 2023), and it is right-skewed (skewness 0.60).

The index drifts sideways and lower from 100 in early 2020 to its low of 89.02 in May 2021, stays mostly in the
90s through the Peak phase, then climbs strongly from 2023 to its high of 146.21 in July 2025. It rises most
during the Slowdown phase (mean 115.53, up from 95.38 at Peak), the opposite of what equity markets usually do
in a downturn, which again suggests the indicators were generated independently of the cycle label.

Like housing prices, it is highly correlated with time (r = 0.91) and with `housing_price_index` (r = 0.93), so
only one of these trending indices should be kept, preferably as a percentage change. Its relevance to
transaction-level AML risk is weak, although it could matter for wealth or investment products.
"""

In [124]:
# Do not modify this code
print_tile(size="h3", key='feature_10_insights', value=feature_10_insights)

### B.12 Explore Feature of Interest `recession_flag`

> You can add more cells in this section

In [125]:
recession_flag_df = df['recession_flag']

In [126]:
recession_flag_df.describe()

count    72.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: recession_flag, dtype: float64

In [127]:
recession_flag_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 72 entries, 0 to 71
Series name: recession_flag
Non-Null Count  Dtype
--------------  -----
72 non-null     int64
dtypes: int64(1)
memory usage: 708.0 bytes


In [128]:
recession_flag_df.unique()

array([0])

In [129]:
recession_flag_df.isna().sum()

np.int64(0)

In [130]:
categorical_distribution('recession_flag')

alt.Chart(...)

In [131]:
# recession_flag within each economic cycle
pd.crosstab(df['economic_cycle'], df['recession_flag']).reindex(CYCLE_ORDER)

recession_flag,0
economic_cycle,
Expansion,18
Peak,18
Slowdown,18
Recovery,18


In [132]:
# <Student to fill this section>
feature_11_insights = """
`recession_flag` is a binary indicator of whether the month is in recession, and it is 0 in all 72 months, with no
missing values. Its mean and standard deviation are both 0.

A constant column carries no information for analysis or modelling and should be dropped. It is also
inconsistent with the other indicators: it stays at 0 through the 18-month Slowdown phase, when
`consumer_sentiment` falls to around 70 and `loan_stress_index` reaches its highest levels, and through 2020,
when Australia experienced its first recession in nearly 30 years.

If a downturn indicator is needed, it would be better to derive one from `economic_cycle` (e.g. Slowdown = 1) or
from a threshold on `consumer_sentiment` or `loan_stress_index`, though these would share the time-proxy issue
noted for the cycle label.
"""

In [133]:
# Do not modify this code
print_tile(size="h3", key='feature_11_insights', value=feature_11_insights)

## C. Ethical Consideration of Features


In [134]:
# <Student to fill this section and then remove this comment>

In [135]:
# <Student to fill this section and then remove this comment>
ethical_considerations = """
Explain ethical concerns related to the selected features
"""

In [136]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)